In [1]:
!pip install -q transformers sentence-transformers faiss-cpu gradio pandas torch accelerate bitsandbytes

print("✅ All libraries installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.0 MB/s eta 0:00:00
✅ All libraries installed!


In [2]:
from google.colab import files
import pandas as pd

df = pd.read_csv('hadith_bukhari_muslim_complete.csv')

print(f"\nLoaded {len(df):,} hadiths")
print(f"Collections: {df['collection'].unique().tolist()}")


Loaded 14,372 hadiths
Collections: ['Sahih Muslim', 'Sahih al-Bukhari']


In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import json
import re

class EnhancedHadithRAG:
    """
    Enhanced RAG System with Boundary Case Handling

    NEW FEATURES:
    1. Query validation (Islam-related check)
    2. Relevance threshold filtering
    3. Chain-only hadith detection
    4. Query expansion for better understanding
    """

    def __init__(self, dataframe):
        print("Initializing Enhanced Hadith RAG System\n")
        print("="*70)

        self.df = dataframe
        print(f"Database: {len(self.df):,} hadiths")

        # RETRIEVAL: Embedding model
        print("\n[1/3] Loading retrieval model...")
        self.embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
        print("      Retrieval model ready")

        # GENERATION: Qwen2.5-7B
        print("\n[2/3] Loading generation model (Qwen2.5-7B)...")
        print("      This may take 3-5 minutes...")

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )

        model_name = "Qwen/Qwen2.5-7B-Instruct"

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
        self.model.eval()

        print("      Generation model ready")

        # Build index
        print("\n[3/3] Building search index...")
        self._build_index()

        # Islamic keywords for validation
        self._islamic_keywords = {
            'islam', 'muslim', 'allah', 'prophet', 'muhammad', 'hadith', 'quran',
            'prayer', 'salah', 'namaz', 'zakah', 'charity', 'fasting', 'ramadan',
            'hajj', 'pilgrimage', 'faith', 'iman', 'worship', 'dua', 'mosque',
            'companion', 'sahaba', 'caliph', 'imam', 'shariah', 'halal', 'haram',
            'sunnah', 'believers', 'righteousness', 'paradise', 'hell', 'judgment'
        }

        print("\n" + "="*70)
        print("Enhanced RAG System Ready")
        print("="*70 + "\n")

    def _build_index(self):
        """Build FAISS index"""

        texts = self.df['hadith_text_en'].tolist()

        self.embeddings = self.embedder.encode(
            texts,
            batch_size=64,
            show_progress_bar=True,
            convert_to_numpy=True
        )

        dimension = self.embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dimension)
        self.index.add(self.embeddings.astype('float32'))

        print(f"      Index built: {self.index.ntotal:,} hadiths")

    def _is_islamic_query(self, query):
        """
        NEW: Validate if query is Islam-related

        Returns: (is_valid, confidence)
        """

        query_lower = query.lower()

        # Check for Islamic keywords
        keyword_matches = sum(1 for keyword in self._islamic_keywords if keyword in query_lower)

        if keyword_matches > 0:
            return True, "keyword_match"

        # Check for common Islamic question patterns
        islamic_patterns = [
            r'what (does|do) islam',
            r'islamic (view|perspective|teaching)',
            r'in islam',
            r'according to (islam|hadith|quran)',
            r'prophet.*said',
            r'messenger of allah',
        ]

        for pattern in islamic_patterns:
            if re.search(pattern, query_lower):
                return True, "pattern_match"

        # If no clear Islamic indicators, use semantic similarity
        # Check if query is semantically close to Islamic topics
        islamic_reference_queries = [
            "What does Islam teach?",
            "Islamic guidance on this topic",
            "What did the Prophet say?"
        ]

        query_vec = self.embedder.encode([query])
        ref_vecs = self.embedder.encode(islamic_reference_queries)

        # Calculate similarity with Islamic reference queries
        similarities = []
        for ref_vec in ref_vecs:
            sim = np.dot(query_vec[0], ref_vec) / (np.linalg.norm(query_vec[0]) * np.linalg.norm(ref_vec))
            similarities.append(sim)

        max_similarity = max(similarities)

        # If query is somewhat related to Islamic topics
        if max_similarity > 0.3:  # Lower threshold for fuzzy matching
            return True, "semantic_match"

        return False, "not_islamic"

    def _is_chain_only_hadith(self, hadith_text):
        """
        NEW: Detect if hadith contains only chain of narration

        Indicators:
        - Very short text (<100 chars)
        - Contains phrases like "same was conveyed", "narrated through"
        - Multiple names with "from...from...from"
        - No actual teaching content
        """

        text_lower = hadith_text.lower()

        # Check length - chain-only hadiths are usually short
        if len(hadith_text) < 100:
            # Check for chain-only indicators
            chain_indicators = [
                'same',
                'like it',
                'similar',
                'transmitted through',
                'narrated through',
                'conveyed from',
                'reported on the authority of',
            ]

            has_chain_words = sum(1 for indicator in chain_indicators if indicator in text_lower)

            # Check for multiple "from" (indicates chain)
            from_count = text_lower.count(' from ')

            # If short + chain indicators + multiple "from"
            if has_chain_words >= 2 or from_count >= 3:
                return True

        # Check for very short hadiths that are just references
        if len(hadith_text) < 80 and ('narrated' in text_lower or 'reported' in text_lower):
            # Count meaningful words (not just names and chain words)
            meaningful_word_count = len([
                word for word in hadith_text.split()
                if len(word) > 3 and word.lower() not in ['from', 'narrated', 'reported', 'said', 'that']
            ])

            if meaningful_word_count < 10:
                return True

        return False

    def _calculate_relevance_score(self, distance):
        """
        NEW: Calculate relevance score from FAISS distance

        Returns: score between 0-100
        Lower distance = higher relevance
        """

        # FAISS L2 distance typically ranges 0-4 for similar items
        # Convert to 0-100 scale
        score = max(0, 100 - (distance * 25))
        return score

    def _detect_intent(self, query):
        """Enhanced intent detection"""

        query_lower = query.lower().strip()

        # Check for hadith number patterns
        if re.match(r'^(bukhari|muslim)\s+\d+$', query_lower):
            parts = query_lower.split()
            return 'id', parts[0], parts[1]

        if 'bukhari' in query_lower or 'muslim' in query_lower:
            numbers = re.findall(r'\d+', query)
            if numbers:
                collection = 'bukhari' if 'bukhari' in query_lower else 'muslim'
                return 'id', collection, numbers[0]

        return 'topic', None, None

    def _retrieve_by_id(self, collection, number):
        """Direct database lookup"""

        col_map = {
            'bukhari': 'Sahih al-Bukhari',
            'muslim': 'Sahih Muslim'
        }

        collection_name = col_map.get(collection.lower(), collection)

        result = self.df[
            (self.df['collection'] == collection_name) &
            (self.df['hadith_number'] == int(number))
        ]

        if len(result) == 0:
            return None

        h = result.iloc[0]

        # Check if it's a chain-only hadith
        if self._is_chain_only_hadith(str(h['hadith_text_en'])):
            return {
                'type': 'chain_only',
                'collection': str(h['collection']),
                'hadith_number': int(h['hadith_number']),
                'reference': str(h['reference_short']),
                'text': str(h['hadith_text_en'])
            }

        return {
            'collection': str(h['collection']),
            'hadith_number': int(h['hadith_number']),
            'reference': str(h['reference_short']),
            'book': str(h['book_name_en']),
            'text': str(h['hadith_text_en'])
        }

    def _retrieve_by_topic(self, query, top_k=5):
        """
        Enhanced semantic search with quality filtering

        Changes:
        - Retrieve top 5 instead of 3
        - Filter out chain-only hadiths
        - Apply relevance threshold
        - Return top 3 QUALITY results
        """

        query_vec = self.embedder.encode([query])

        # Search top 10 to have buffer for filtering
        distances, indices = self.index.search(query_vec.astype('float32'), k=10)

        results = []
        for idx, dist in zip(indices[0], distances[0]):
            h = self.df.iloc[idx]

            # Calculate relevance score
            relevance = self._calculate_relevance_score(dist)

            # Skip if relevance too low (threshold: 40%)
            if relevance < 40:
                continue

            hadith_text = str(h['hadith_text_en'])

            # Skip chain-only hadiths
            if self._is_chain_only_hadith(hadith_text):
                continue

            results.append({
                'collection': str(h['collection']),
                'hadith_number': int(h['hadith_number']),
                'reference': str(h['reference_short']),
                'book': str(h['book_name_en']),
                'text': hadith_text,
                '_relevance': float(relevance)
            })

            # Stop when we have 3 quality results
            if len(results) >= 3:
                break

        return results

    def _generate_explanation(self, hadith_text):
        """Generate explanation"""

        text_snippet = hadith_text[:650] if len(hadith_text) > 650 else hadith_text

        system_prompt = """You are a hadith presentation assistant.

STRICT RULES:
- Explain ONLY what is stated in the provided hadith text
- Do NOT authenticate or grade hadiths
- Do NOT issue religious rulings (fatwa)
- Do NOT add external interpretations
- Keep explanations brief and factual (2-3 sentences maximum)
- If something requires scholarly interpretation, state that clearly"""

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Provide a brief explanation of this hadith:\n\n{text_snippet}"}
        ]

        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=120,
                temperature=0.3,
                top_p=0.85,
                do_sample=False,
                num_beams=1
            )

        explanation = self.tokenizer.decode(
            outputs[0][len(inputs.input_ids[0]):],
            skip_special_tokens=True
        )

        return explanation.strip()

    def _get_safe_authenticity_note(self, collection):
        """Safe authenticity phrasing"""

        if 'Bukhari' in collection:
            return "This hadith is recorded in Ṣaḥīḥ al-Bukhārī, a collection accepted by Sunni scholars as authentic."
        elif 'Muslim' in collection:
            return "This hadith is recorded in Ṣaḥīḥ Muslim, a collection accepted by Sunni scholars as authentic."
        else:
            return f"This hadith is recorded in {collection}."

    def query(self, user_input):
        """
        Enhanced RAG pipeline with validation
        """

        # STEP 1: Intent detection
        intent_type, collection, number = self._detect_intent(user_input)

        # STEP 2: For topic searches, validate if Islamic query
        if intent_type == 'topic':
            is_valid, reason = self._is_islamic_query(user_input)

            if not is_valid:
                return {
                    'type': 'out_of_scope',
                    'message': 'This query does not appear to be related to Islamic topics. This system searches hadiths from Sahih al-Bukhari and Sahih Muslim. Please ask questions about Islamic teachings, practices, or search for specific hadiths.'
                }

        # STEP 3: Retrieval
        if intent_type == 'id':
            hadith = self._retrieve_by_id(collection, number)

            if not hadith:
                return {
                    'type': 'error',
                    'message': f'Hadith {collection.capitalize()} {number} not found in database.'
                }

            # Check if chain-only
            if hadith.get('type') == 'chain_only':
                return {
                    'type': 'chain_only',
                    'message': f'**{hadith["reference"]}** contains only a chain of narration without the full hadith text.\n\n**Text:** {hadith["text"]}\n\nThis appears to be a variant narration that refers to another hadith in the same chapter. Please try searching by topic or check adjacent hadith numbers for the complete narration.'
                }

            # Generate explanation
            explanation = self._generate_explanation(hadith['text'])

            return {
                'type': 'direct',
                'hadith': hadith,
                'explanation': explanation,
                'authenticity_note': self._get_safe_authenticity_note(hadith['collection'])
            }

        else:  # Topic search
            results = self._retrieve_by_topic(user_input, top_k=3)

            if not results:
                return {
                    'type': 'no_results',
                    'message': f'No relevant hadiths found for "{user_input}". This could mean:\n\n1. The topic may not be directly addressed in Bukhari or Muslim\n2. Try rephrasing with different keywords\n3. The query might be too specific\n\nTry broader terms like "prayer", "charity", "kindness", etc.'
                }

            main_hadith = results[0]

            # Generate explanation
            explanation = self._generate_explanation(main_hadith['text'])

            # Prepare related hadiths
            related = [
                {
                    'reference': r['reference'],
                    'text_preview': r['text'][:120] + '...' if len(r['text']) > 120 else r['text'],
                    'relevance': r['_relevance']
                }
                for r in results[1:]
            ]

            return {
                'type': 'search',
                'main_hadith': main_hadith,
                'explanation': explanation,
                'related': related,
                'authenticity_note': self._get_safe_authenticity_note(main_hadith['collection']),
                'relevance_score': main_hadith['_relevance']
            }

# Initialize Enhanced RAG System
print("="*70)
print("INITIALIZING ENHANCED HADITH RAG SYSTEM")
print("="*70 + "\n")

rag_system = EnhancedHadithRAG(df)

print("SYSTEM READY WITH BOUNDARY CASE HANDLING")

INITIALIZING ENHANCED HADITH RAG SYSTEM

Initializing Enhanced Hadith RAG System

Database: 14,372 hadiths

[1/3] Loading retrieval model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (18 outputs left out)

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

      Generation model ready

[3/3] Building search index...


Batches:   0%|          | 0/225 [00:00<?, ?it/s]

      Index built: 14,372 hadiths

Enhanced RAG System Ready

SYSTEM READY WITH BOUNDARY CASE HANDLING


## User-Friendly Chat Interface

In [5]:

import gradio as gr

def format_response(result):
    """Enhanced formatting with better error handling"""

    # OUT OF SCOPE (non-Islamic query)
    if result['type'] == 'out_of_scope':
        return f"""**❌ Query Out of Scope**

{result['message']}

**Examples of valid queries:**
- "Bukhari 1"
- "What does Islam say about charity?"
- "importance of prayer"
- "Muslim 567"
"""

    # NO RESULTS (Islamic query but no matches)
    if result['type'] == 'no_results':
        return f"""**⚠️ No Results Found**

{result['message']}
"""

    # CHAIN ONLY HADITH
    if result['type'] == 'chain_only':
        return f"""**ℹ️ Chain-Only Narration**

{result['message']}
"""

    # ERROR
    if result['type'] == 'error':
        return f"""**❌ Error**

{result['message']}
"""

    # SUCCESSFUL RESPONSE
    hadith = result.get('hadith') or result.get('main_hadith')

    response = f"""**Hadith Reference**

**Collection:** {hadith['collection']}
**Hadith Number:** {hadith['hadith_number']}
**Book:** {hadith['book']}
"""

    # Show relevance score for topic searches
    if result.get('relevance_score'):
        response += f"**Relevance:** {result['relevance_score']:.1f}%\n"

    response += f"""
---

**Hadith Text**

{hadith['text']}

---

**Brief Explanation**

{result['explanation']}

---

**Authenticity Note**

{result['authenticity_note']}

---

**Reference:** {hadith['reference']}
"""

    # Add related hadiths
    if result.get('related') and len(result['related']) > 0:
        response += "\n\n**Related Narrations**\n\n"
        for rel in result['related']:
            response += f"- **{rel['reference']}** (Relevance: {rel['relevance']:.1f}%)\n"
            response += f"  _{rel['text_preview']}_\n\n"

    # Educational disclaimer
    response += "\n---\n\n_Explanations are provided for educational purposes based on the wording of the hadith._"

    return response

def chat_function(message, history):
    """Process chat with enhanced validation"""

    if not message.strip():
        return history, ""

    # Add user message immediately
    history = history + [[message, None]]

    # Get RAG response
    try:
        result = rag_system.query(message)
        bot_response = format_response(result)
        history[-1][1] = bot_response

    except Exception as e:
        history[-1][1] = f"**❌ System Error**\n\nFailed to process query: {str(e)}\n\nPlease try again or rephrase your question."

    return history, ""

# Same CSS as before
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600&display=swap');

* {
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important;
}

:root {
    --bg-primary: #0a0f0d;
    --bg-secondary: #1a2520;
    --bg-card: #243329;
    --accent-green: #2d6a4f;
    --accent-light: #52b788;
    --text-primary: #ffffff;
    --text-secondary: #b7e4c7;
    --border: #2d6a4f;
}

.gradio-container {
    background: var(--bg-primary) !important;
}

.contain {
    max-width: 1000px !important;
    margin: 0 auto !important;
}

#chatbot-container {
    height: 600px !important;
    display: flex !important;
    flex-direction: column !important;
}

#chatbot {
    flex: 1 !important;
    background: var(--bg-secondary) !important;
    border: 2px solid var(--border) !important;
    border-radius: 12px !important;
    overflow-y: auto !important;
}

.message {
    padding: 16px 20px !important;
    margin: 12px 0 !important;
    border-radius: 8px !important;
    line-height: 1.6 !important;
}

.user.message {
    background: var(--accent-green) !important;
    color: white !important;
    margin-left: 20% !important;
}

.bot.message {
    background: var(--bg-card) !important;
    color: var(--text-primary) !important;
    border: 1px solid var(--border) !important;
    margin-right: 10% !important;
}

#input-row {
    position: sticky !important;
    bottom: 0 !important;
    background: var(--bg-primary) !important;
    padding: 16px 0 !important;
    border-top: 2px solid var(--border) !important;
    margin-top: 16px !important;
}

input, textarea {
    background: var(--bg-secondary) !important;
    border: 2px solid var(--border) !important;
    color: var(--text-primary) !important;
    border-radius: 8px !important;
    padding: 12px 16px !important;
}

input:focus, textarea:focus {
    border-color: var(--accent-light) !important;
    outline: none !important;
}

button {
    background: var(--accent-green) !important;
    border: none !important;
    color: white !important;
    border-radius: 8px !important;
    font-weight: 500 !important;
    padding: 12px 24px !important;
    transition: all 0.2s !important;
}

button:hover {
    background: var(--accent-light) !important;
    transform: translateY(-1px) !important;
}

h1, h2, h3 {
    color: var(--accent-light) !important;
}

code {
    background: var(--bg-primary) !important;
    color: var(--accent-light) !important;
    padding: 2px 6px !important;
    border-radius: 4px !important;
}

footer {
    display: none !important;
}

::-webkit-scrollbar {
    width: 8px;
}

::-webkit-scrollbar-track {
    background: var(--bg-primary);
}

::-webkit-scrollbar-thumb {
    background: var(--accent-green);
    border-radius: 4px;
}

::-webkit-scrollbar-thumb:hover {
    background: var(--accent-light);
}
"""

# CREATE INTERFACE
with gr.Blocks(css=custom_css, title="HadithRAG", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # HadithRAG

    **Retrieval-Augmented Generation for Hadith Explanation**

    Search by hadith number (e.g., "Bukhari 1", "Muslim 567") or topic (e.g., "charity", "prayer importance")

    **Database:** 14,372 hadiths from Ṣaḥīḥ al-Bukhārī and Ṣaḥīḥ Muslim
    **Features:** Relevance filtering, chain-only detection, query validation
    """)

    with gr.Column(elem_id="chatbot-container"):
        chatbot = gr.Chatbot(
            label="",
            show_label=False,
            height=550,
            elem_id="chatbot",
            avatar_images=None,
            show_copy_button=True
        )

    with gr.Row(elem_id="input-row"):
        msg = gr.Textbox(
            placeholder="Enter hadith number (e.g., Muslim 567) or search by topic...",
            show_label=False,
            scale=5,
            container=False
        )
        send = gr.Button("Send", scale=1, variant="primary")

    clear = gr.ClearButton([msg, chatbot], value="Clear Chat", size="sm")

    gr.Examples(
        examples=[
            "Bukhari 1",
            "Muslim 567",
            "What does Islam say about charity?",
            "importance of prayer",
            "kindness to parents",
            "Who is Elon Musk?",  # Will be rejected
        ],
        inputs=msg,
        label="Example Queries (try the last one to see validation)"
    )

    gr.Markdown("""
    ---

    **System Features:**
    - ✅ Query validation (filters non-Islamic queries)
    - ✅ Relevance threshold (minimum 40% similarity)
    - ✅ Chain-only detection (skips hadiths with just narrator chains)
    - ✅ Quality filtering (returns meaningful content only)

    **Note:** This system retrieves hadiths from authenticated collections and provides explanations based solely on the hadith text. For religious rulings, consult qualified Islamic scholars.

    **Educational Disclaimer:** Explanations are provided for educational purposes based on the wording of the hadith.
    """)

    send.click(chat_function, [msg, chatbot], [chatbot, msg])
    msg.submit(chat_function, [msg, chatbot], [chatbot, msg])

print("\n" + "="*70)
print("LAUNCHING ENHANCED CHAT INTERFACE")
print("="*70 + "\n")

demo.launch(share=True, debug=False, show_error=True)

print("\n✅ Enhanced Interface Live!")
print("📋 Now handles boundary cases properly")

/tmp/ipython-input-2224902836.py:243: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, title="HadithRAG", theme=gr.themes.Soft()) as demo:
/tmp/ipython-input-2224902836.py:243: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, title="HadithRAG", theme=gr.themes.Soft()) as demo:
/tmp/ipython-input-2224902836.py:257: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipython-input-2224902836.py:257: DeprecationWarning: The 'show_copy_butt


LAUNCHING ENHANCED CHAT INTERFACE

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6e57258afa13ce85f1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



✅ Enhanced Interface Live!
📋 Now handles boundary cases properly
